# 🧾 Quotation Intelligence — API & LLM Test Notebook

This notebook tests:
1. **LiteLLM direct** — call any LLM provider directly
2. **Health endpoints** — verify the API is running
3. **Standalone extract** — PDF → structured data (no DB / Redis / Celery)
4. **Full pipeline** — upload → queue → poll → export (requires PostgreSQL + Redis)
5. **Export endpoints** — download JSON / CSV / Excel

---
### Prerequisites
```bash
# Terminal 1 — start API
poetry run uvicorn quotation_intelligence.api.main:app --reload

# Terminal 2 — install notebook kernel (once)
poetry run pip install ipykernel requests
```

> Set `PDF_PATH` in the **Config** cell below before running.

---
## 0. Config & Helpers

In [ ]:
# !poetry run pip install ipykernel Request

In [ ]:
import base64
import json
import os
import sys
import time
from pathlib import Path

import requests

# ── CONFIG — edit these ────────────────────────────────────────────────────
BASE_URL   = "http://127.0.0.1:8000"
API_KEY    = ""                          # leave blank in dev mode
PDF_PATH   = r"C:/Users/ASUS LAPTOP/Downloads/Quotation - S06666 (1).pdf"  # ← change me

# LiteLLM / LLM settings (reads from .env automatically, but you can override here)
LLM_MODEL="ollama/gemma4:e2b"  # or ollama/llama3.2, openai/gpt-4o …
# ──────────────────────────────────────────────────────────────────────────

HEADERS = {"Content-Type": "application/json", "accept": "application/json"}
if API_KEY:
    HEADERS["X-API-Key"] = API_KEY

def pretty(response: requests.Response) -> None:
    """Pretty-print an HTTP response."""
    print(f"Status : {response.status_code}")
    try:
        print(json.dumps(response.json(), indent=2, default=str))
    except Exception:
        print(response.text[:2000])

print("Config loaded ✅")
print(f"  API  : {BASE_URL}")
print(f"  PDF  : {PDF_PATH}")
print(f"  Model: {LLM_MODEL}")

---
## 1. 🤖 LiteLLM — Direct LLM Test
Tests the LLM gateway directly, **without** the HTTP API.

In [ ]:
# Add the project root to sys.path so we can import quotation_intelligence
PROJECT_ROOT = str(Path.cwd().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import litellm
from dotenv import load_dotenv

load_dotenv(Path(PROJECT_ROOT) / ".env")   # load API keys from .env

litellm.suppress_debug_info = True

response = litellm.completion(
    model=LLM_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Reply concisely."},
        {"role": "user",   "content": "What is a quotation document? Answer in 2 sentences."},
    ],
    # max_tokens=150,
    # temperature=0.0,
    api_base="http://localhost:11434",
)
print(response)
print("✅ LLM response received")
print(f"  Model    : {response.model}")
print(f"  Tokens   : {response.usage}")
print(f"  Answer   : {response.choices[0].message.content}")

In [ ]:
from litellm import completion

response = completion(
    model="ollama/gemma4:e4b", 
    # messages=[{ "content": "respond in 20 words. who are you?","role": "user"}], 
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Reply concisely."},
        {"role": "user",   "content": "What is a quotation document? Answer in 2 sentences."},
    ],
    api_base="http://localhost:11434"
)
print(response.choices[0].message.content)


In [ ]:
# ── Test: LLM extraction service directly (no HTTP, no DB) ─────────────────
from quotation_intelligence.extraction.llm_service import LLMService

service = LLMService()

sample_text = """
QUOTATION
Supplier: Acme Electronics Ltd
Quote No: Q-2024-0891
Date: 15 January 2024
Currency: USD

Item  Description              Qty   Unit Price   Total
1     Laptop Model X500        10    $899.00      $8,990.00
2     Wireless Mouse WM-200    20    $29.99       $599.80
3     USB-C Hub 7-Port         15    $49.95       $749.25

Subtotal:  $10,339.05
Tax (10%): $1,033.91
Total:     $11,372.96
"""

result = service.extract_quotation(sample_text)

print("✅ Extraction complete")
print(f"  Supplier        : {result.supplier_name} (conf: {result.supplier_name_confidence:.2f})")
print(f"  Quotation No.   : {result.quotation_number} (conf: {result.quotation_number_confidence:.2f})")
print(f"  Date            : {result.quotation_date}")
print(f"  Currency        : {result.currency}")
print(f"  Total Amount    : {result.total_amount} (conf: {result.total_confidence:.2f})")
print(f"  Line Items      : {len(result.line_items)}")
print(f"  Overall Conf.   : {result.get_overall_confidence():.3f}")
print(f"  Missing Fields  : {result.get_missing_fields()}")
print("\nLine Items:")
for item in result.line_items:
    print(f"  [{item.line_number}] {item.description} | qty={item.quantity} | unit=${item.unit_price} | total=${item.total_price}")

---
## 2. ❤️ Health Check Endpoints

In [ ]:
# Basic health check
resp = requests.get(f"{BASE_URL}/health")
print("GET /health")
pretty(resp)

In [ ]:
# Detailed health (v1)
resp = requests.get(f"{BASE_URL}/api/v1/health", headers=HEADERS)
print("GET /api/v1/health")
pretty(resp)

---
## 3. 🚀 Standalone Extract (No DB / Redis / Celery)
Results returned **immediately** — ideal for testing.

In [ ]:
# ── Option A: Local file path ──────────────────────────────────────────────
payload = {
    "local_upload": {
        "file_path": PDF_PATH,
        "metadata": {"source": "notebook_test"}
    }
}

print(f"POST /api/v1/standalone/extract  (local file)")
print(f"  File: {PDF_PATH}")
t0 = time.time()

resp = requests.post(
    f"{BASE_URL}/api/v1/standalone/extract",
    headers=HEADERS,
    json=payload,
    timeout=500,
)

print(f"  Took : {time.time()-t0:.1f}s")
pretty(resp)

In [ ]:
# Store result for further inspection
if resp.status_code == 200:
    extraction = resp.json()
    data = extraction["data"]
    summary = extraction["confidence_summary"]

    print("\n── Extraction Summary ──────────────────────────────────")
    print(f"  Status           : {extraction['status']}")
    print(f"  Processing time  : {extraction['processing_time_seconds']}s")
    print(f"  Supplier         : {data.get('supplier_name')}")
    print(f"  Quotation No.    : {data.get('quotation_number')}")
    print(f"  Date             : {data.get('quotation_date')}")
    print(f"  Currency         : {data.get('currency')}")
    print(f"  Total Amount     : {data.get('total_amount')}")
    print(f"  Line Items       : {summary['line_item_count']}")
    print(f"  Avg Confidence   : {summary['average_confidence']:.3f}")
    print(f"  High conf items  : {summary['high_confidence_count']}")
    print(f"  Missing fields   : {summary['missing_fields']}")

    print("\n── Line Items ──────────────────────────────────────────")
    for item in data.get("line_items", []):
        print(f"  [{item.get('line_number','?')}] {str(item.get('description',''))[:50]:50s} "
              f"qty={item.get('quantity')}  unit=${item.get('unit_price')}  total=${item.get('total_price')}")
else:
    print("Request failed — check the error above")

In [ ]:
# ── Option B: Base64 upload ────────────────────────────────────────────────
pdf_path = Path(PDF_PATH)

if not pdf_path.exists():
    print(f"⚠️  File not found: {PDF_PATH}")
else:
    b64_content = base64.b64encode(pdf_path.read_bytes()).decode()

    payload_b64 = {
        "base64_upload": {
            "file_name": pdf_path.name,
            "file_content_base64": b64_content,
            "metadata": {"source": "notebook_base64_test"}
        }
    }

    print(f"POST /api/v1/standalone/extract  (base64, {len(b64_content):,} chars)")
    t0 = time.time()

    resp_b64 = requests.post(
        f"{BASE_URL}/api/v1/standalone/extract",
        headers=HEADERS,
        json=payload_b64,
        timeout=120,
    )

    print(f"  Took : {time.time()-t0:.1f}s")
    pretty(resp_b64)

---
## 4. 📥 Full Pipeline (requires PostgreSQL + Redis + Celery)

> Start the Celery worker first:
> ```bash
> poetry run celery -A quotation_intelligence.tasks worker --loglevel=info
> ```

In [ ]:
# Step 1 — Upload document
payload_upload = {
    "local_upload": {
        "file_path": PDF_PATH,
        "metadata": {"source": "notebook_full_pipeline"}
    }
}

resp_upload = requests.post(
    f"{BASE_URL}/api/v1/documents/upload",
    headers=HEADERS,
    json=payload_upload,
    timeout=30,
)

print("POST /api/v1/documents/upload")
pretty(resp_upload)

if resp_upload.status_code == 202:
    DOCUMENT_ID = resp_upload.json()["document_id"]
    print(f"\n  Document ID: {DOCUMENT_ID}")
else:
    DOCUMENT_ID = None
    print("\n⚠️  Upload failed (DB/Redis/Celery may not be running)")

In [ ]:
# Step 2 — Poll until completed
if DOCUMENT_ID:
    for attempt in range(12):   # poll up to 60s
        resp_status = requests.get(
            f"{BASE_URL}/api/v1/documents/{DOCUMENT_ID}",
            headers=HEADERS,
        )
        doc = resp_status.json()
        status = doc.get("status", "unknown")
        print(f"  [{attempt+1:2d}] status = {status}")

        if status in ("completed", "partial", "failed"):
            break
        time.sleep(5)

    print("\nFinal document response:")
    pretty(resp_status)
else:
    print("Skipped — no document_id (upload failed)")

In [ ]:
# Step 3 — List all documents
resp_list = requests.get(
    f"{BASE_URL}/api/v1/documents/",
    headers=HEADERS,
    params={"limit": 5},
)
print("GET /api/v1/documents/")
pretty(resp_list)

In [ ]:
# Step 4 — Reprocess a document
if DOCUMENT_ID:
    resp_reprocess = requests.post(
        f"{BASE_URL}/api/v1/documents/{DOCUMENT_ID}/reprocess",
        headers=HEADERS,
    )
    print(f"POST /api/v1/documents/{DOCUMENT_ID}/reprocess")
    pretty(resp_reprocess)
else:
    print("Skipped — no document_id")

---
## 5. 📤 Export Endpoints
Requires a completed document. Uses `DOCUMENT_ID` from Step 4, or set it manually below.

In [ ]:
# Set manually if not running the full pipeline above
# DOCUMENT_ID = "paste-uuid-here"

if not DOCUMENT_ID:
    print("⚠️  Set DOCUMENT_ID to a completed document UUID")
else:
    # Export JSON
    resp_json = requests.get(
        f"{BASE_URL}/api/v1/exports/{DOCUMENT_ID}",
        headers=HEADERS,
        params={"format": "json"},
    )
    print("GET /api/v1/exports/{id}?format=json")
    pretty(resp_json)

In [ ]:
# Export CSV — save to file
if DOCUMENT_ID:
    resp_csv = requests.get(
        f"{BASE_URL}/api/v1/exports/{DOCUMENT_ID}",
        headers=HEADERS,
        params={"format": "csv"},
    )
    out_csv = Path("output_quotation.csv")
    out_csv.write_bytes(resp_csv.content)
    print(f"✅ CSV saved to {out_csv.resolve()} ({len(resp_csv.content):,} bytes)")
    print(resp_csv.content.decode()[:800])

In [ ]:
# Export Excel — save to file
if DOCUMENT_ID:
    resp_xlsx = requests.get(
        f"{BASE_URL}/api/v1/exports/{DOCUMENT_ID}",
        headers=HEADERS,
        params={"format": "excel"},
    )
    out_xlsx = Path("output_quotation.xlsx")
    out_xlsx.write_bytes(resp_xlsx.content)
    print(f"✅ Excel saved to {out_xlsx.resolve()} ({len(resp_xlsx.content):,} bytes)")

In [ ]:
# Preview endpoint
if DOCUMENT_ID:
    resp_preview = requests.get(
        f"{BASE_URL}/api/v1/exports/preview/{DOCUMENT_ID}",
        headers=HEADERS,
    )
    print("GET /api/v1/exports/preview/{id}")
    pretty(resp_preview)

---
## 6. 🔬 Extraction Pipeline — Unit Test (no HTTP)
Tests each pipeline component individually.

In [ ]:
# Test: PDF Parser
from quotation_intelligence.extraction.pdf_parser import PDFParser

parser = PDFParser(ocr_enabled=False)
pdf_path = Path(PDF_PATH)

if pdf_path.exists():
    pages = parser.parse(pdf_path)
    full_text = parser.get_full_text(pages)
    metadata = parser.extract_metadata(pdf_path)

    print(f"✅ PDF parsed")
    print(f"  Pages    : {len(pages)}")
    print(f"  Metadata : {metadata}")
    print(f"  Text len : {len(full_text):,} chars")
    print(f"\nFirst 500 chars:\n{full_text[:500]}")
else:
    print(f"⚠️  PDF not found: {PDF_PATH}")

In [ ]:
print(full_text)

In [ ]:
# Test: Regex Extractor
from quotation_intelligence.extraction.regex_extractor import RegexExtractor

extractor = RegexExtractor()

if 'full_text' in dir() and full_text:
    candidates = extractor.extract(full_text)
    print(f"✅ Regex extraction complete — {len(candidates)} field(s) found")
    for field, matches in candidates.items():
        best = matches[0]
        print(f"  {field:25s}: {str(best.value)[:40]:40s}  (conf={best.confidence:.2f})")
else:
    # Fallback to sample text
    sample = "Quote No: Q-2024-0891  Date: 15/01/2024  Total: $11,372.96"
    candidates = extractor.extract(sample)
    print(f"✅ Regex on sample text — {len(candidates)} field(s) found")
    for field, matches in candidates.items():
        print(f"  {field}: {matches[0].value} (conf={matches[0].confidence:.2f})")

In [ ]:
# Test: Full sync pipeline (no DB)
from quotation_intelligence.extraction.pipeline import ExtractionPipeline

pdf_path = Path(PDF_PATH)

if pdf_path.exists():
    pipeline = ExtractionPipeline()
    t0 = time.time()
    result = pipeline.process_sync(pdf_path)
    elapsed = time.time() - t0

    print(f"✅ Pipeline complete in {elapsed:.1f}s")
    print(f"  Supplier      : {result.supplier_name}")
    print(f"  Quotation No. : {result.quotation_number}")
    print(f"  Date          : {result.quotation_date}")
    print(f"  Currency      : {result.currency}")
    print(f"  Total         : {result.total_amount}")
    print(f"  Line items    : {len(result.line_items)}")
    print(f"  Confidence    : {result.get_overall_confidence():.3f}")
    print(f"  Missing       : {result.get_missing_fields()}")
    print(f"  Errors        : {result.extraction_errors}")
else:
    print(f"⚠️  PDF not found: {PDF_PATH}")

---
## 7. 🔄 LiteLLM Provider Switching Demo
Shows how to switch providers without changing code — just change the model string.

In [ ]:
import litellm
litellm.suppress_debug_info = True

PROMPT = [
    {"role": "user", "content": "Say 'Hello from <provider>!' and nothing else."}
]

# ── Test each provider you have configured ─────────────────────────────────
providers_to_test = [
    # (model_string,                        api_base or None)
    ("anthropic/claude-3-5-sonnet-20241022", None),
    # ("openai/gpt-4o-mini",                None),
    # ("groq/llama3-70b-8192",              None),
    # ("ollama/llama3.2",                   "http://localhost:11434"),
    # ("gemini/gemini-1.5-flash",           None),
]

for model, api_base in providers_to_test:
    try:
        kwargs = dict(model=model, messages=PROMPT, max_tokens=30, temperature=0.0)
        if api_base:
            kwargs["api_base"] = api_base
        r = litellm.completion(**kwargs)
        print(f"  ✅ {model:45s} → {r.choices[0].message.content.strip()}")
    except Exception as e:
        print(f"  ❌ {model:45s} → {type(e).__name__}: {str(e)[:80]}")

## PDF parsing using PYMUPDF

In [1]:
import fitz
import pymupdf4llm

def extract_quotation(pdf_path: str, data_pages: list = None):
    doc = fitz.open(pdf_path)
    total_pages = doc.page_count

    pages = pymupdf4llm.to_markdown(
        pdf_path,
        page_chunks=True,
        pages=data_pages or list(range(total_pages))
    )

    print(pages)

    md_text = "\n\n".join(
    f"=== PAGE {p['metadata']['page_number']} ===\n{p['text']}"
    for p in pages
    )
    return md_text
md = extract_quotation(r"C:\Users\ASUS LAPTOP\Downloads\Quotation - S06666 (1).pdf")
md

[defaultdict(<function make_page_chunk.<locals>.<lambda> at 0x0000023B79F65B10>, {'metadata': {'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'creator': 'wkhtmltopdf 0.12.6.1', 'producer': 'Qt 4.8.7', 'creationDate': 'D:20260407143223Z', 'modDate': '', 'trapped': '', 'encryption': None, 'file_path': 'C:\\Users\\ASUS LAPTOP\\Downloads\\Quotation - S06666 (1).pdf', 'page_count': 8, 'page_number': 1}, 'toc_items': [[1, 'Quotation # S06666', 1]], 'page_boxes': [{'index': 0, 'class': 'page-header', 'bbox': (405, 30, 569, 98), 'pos': (0, 90)}, {'index': 1, 'class': 'picture', 'bbox': (26, 25, 89, 88), 'pos': (90, 143)}, {'index': 2, 'class': 'text', 'bbox': (26, 151, 300, 175), 'pos': (143, 221)}, {'index': 3, 'class': 'section-header', 'bbox': (26, 198, 212, 217), 'pos': (221, 245)}, {'index': 4, 'class': 'picture', 'bbox': (26, 225, 569, 325), 'pos': (245, 567)}, {'index': 5, 'class': 'table', 'bbox': (26, 326, 569, 716), 'pos': (567, 1118)}, {'index': 6, 'c

"=== PAGE 1 ===\n**ﻟﻠﺘﺠﺎرة ﺗﻤﺎﻳﻞ ﺷﺮﻛﺔ Trydolight Co. Trading VAT No: 314401329900003 CR No: 1010619001** \n\n**==> picture [63 x 63] intentionally omitted <==**\n\n1023009/LIGHTING TECHNOLOGIES CO LTD / LIGHTECH VAT Number: 300056677400003 \n\n## Quotation # S06666 \n\n**==> picture [543 x 100] intentionally omitted <==**\n\n**----- Start of picture text -----**<br>\nYour Reference Quotation Date Expiration Salesperson<br>S06666 04/06/2026 05/06/2026 Elias Aneza<br>Project Name: Payment Terms Warranty<br>Setraa - Malaga School Project Cash N/A<br>**----- End of picture text -----**<br>\n\n\n|||||\n|---|---|---|---|\n||**Prepared By**|**Checked**<br>**By**|**Approved By**|\n|||||\n|Description||Picture<br>Quantity<br>Unit Price<br>Amount||\n|**girls building part 1**||||\n|[BTN-WP / VIVA / 3370] TDL - V-WP36W65K BATTEN WP<br>36W 6500K,5000lm- IP65 220-240V 1.2M- 50,000Hrs-<br>PHILIPS DRIVER & Chip, SAUDI MADE.<br>324.00 Units<br>92.92<br>30,106.08 SR||||\n|[BTN-WP / VIVA / 3370] TDL - V

In [2]:
# ── Test: LLM extraction service directly (no HTTP, no DB) ─────────────────
from quotation_intelligence.extraction.llm_service import LLMService

service = LLMService()

sample_text = md

result = service.extract_quotation(sample_text)

print(result)

# print("✅ Extraction complete")
# print(f"  Supplier        : {result.supplier_name} (conf: {result.supplier_name_confidence:.2f})")
# print(f"  Quotation No.   : {result.quotation_number} (conf: {result.quotation_number_confidence:.2f})")
# print(f"  Date            : {result.quotation_date}")
# print(f"  Currency        : {result.currency}")
# print(f"  Total Amount    : {result.total_amount} (conf: {result.total_confidence:.2f})")
# print(f"  Line Items      : {len(result.line_items)}")
# print(f"  Overall Conf.   : {result.get_overall_confidence():.3f}")
# print(f"  Missing Fields  : {result.get_missing_fields()}")
# print("\nLine Items:")
# for item in result.line_items:
#     print(f"  [{item.line_number}] {item.description} | qty={item.quantity} | unit=${item.unit_price} | total=${item.total_price}")

2026-04-18 13:37:39 [info     ] llm_service_initialized        api_base=http://localhost:11434 model=ollama/kimi-k2.5:cloud
2026-04-18 13:37:39 [info     ] llm_extraction_started         model=ollama/kimi-k2.5:cloud text_length=11173
```json
{
  "supplier_name": "Trydolight Co. Trading",
  "quotation_number": "S06666",
  "quotation_date": "04/06/2026",
  "currency": "SAR",
  "subtotal": 502359.28,
  "tax_amount": 75353.89,
  "total_amount": 577713.17,
  "supplier_name_confidence": 0.95,
  "quotation_number_confidence": 0.95,
  "quotation_date_confidence": 0.95,
  "total_confidence": 0.95,
  "extraction_errors": [],
  "line_items": [
    {
      "line_number": 1,
      "product_code": "BTN-WP / VIVA / 3370",
      "description": "TDL - V-WP36W65K BATTEN WP 36W 6500K,5000lm- IP65 220-240V 1.2M- 50,000Hrs- PHILIPS DRIVER & Chip, SAUDI MADE.",
      "quantity": 324,
      "unit_of_measure": "Units",
      "unit_price": 92.92,
      "total_price": 30106.08,
      "product_code_confidence": 

In [ ]:
import json
json.loads(result)